# VoxSentinel AI — Scam text classification (MuRIL)
Prepared configuration for VoxSentinel AI. **No model is trained by the project by default.**
Run top-to-bottom in Colab (T4/A100). Every section prints what it actually did.


In [ ]:
!pip -q install "transformers==4.44.*" "datasets==2.20.*" scikit-learn

## Data
A consented in-house transcript corpus is required for Hindi/Hinglish voice fraud. Public SMS corpora are English-only sanity checks and must not be presented as voice-fraud evidence.

In [ ]:
# Realistic minimum: ~4k labelled transcripts across ransom / otp / banking / benign / awareness.
# Splits are stratified BY SCENARIO TEMPLATE so the same script never appears twice.
from sklearn.model_selection import train_test_split
import pandas as pd
df = pd.read_csv("/content/transcripts.csv")   # columns: text,label,scenario_id,lang
train, rest = train_test_split(df, test_size=0.3, stratify=df.label, random_state=42)
val, test = train_test_split(rest, test_size=0.5, stratify=rest.label, random_state=42)
print(len(train), len(val), len(test))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained("google/muril-base-cased")
model = AutoModelForSequenceClassification.from_pretrained("google/muril-base-cased", num_labels=5)
print(model.config.num_labels, "labels: ransom, otp, banking, benign, awareness")

## Safety metric
Beyond macro-F1, always report **strong-warning suppression**: share of true ransom cases where any review stage softened the verdict. It must be ~0 without verbatim evidence.